# NWC 계수(γ) 진단 노트북 (v3)

**v3 변경점**: DATA 경로를 노트북 환경에 맞게 설정 완료

## 실행 순서
1. Cell 1-A: `import config` (경로 이미 설정됨)
2. Cell 1-B: `get_db_info()` 호출 & key 매핑
3. Cell 1-B 끝: DB 접속 테스트
4. Cell 2-6: 진단 실행
5. Cell 7: 해석 가이드

## Cell 1-A · DATA 폴더 경로 자동 감지 & config import

In [1]:
import sys, os

# ── DATA 폴더 후보 경로 ─────────────────────────────────────
#    (첫 번째: 현재 노트북 환경, 나머지: 데스크톱 등 다른 머신)
DATA_CANDIDATES = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",  # ← 현재 노트북
    # 다른 머신(데스크톱 등)의 경로는 여기에 추가
]

data_path = None
for p in DATA_CANDIDATES:
    if os.path.isdir(p):
        data_path = p
        if p not in sys.path:
            sys.path.insert(0, p)
        print(f"[OK] DATA 경로: {p}")
        break

if data_path is None:
    raise RuntimeError(
        "DATA 폴더를 찾지 못했습니다. DATA_CANDIDATES에 실제 경로를 추가하세요."
    )

# ── config import ──────────────────────────────────────────
import config
print(f"[OK] config.py 로드됨: {config.__file__}")
print(f"[OK] get_db_info 존재 여부: {'get_db_info' in dir(config)}")


[OK] DATA 경로: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA


ModuleNotFoundError: No module named 'DATA'

## Cell 1-B · `get_db_info()` 호출 & pymysql 호환 변환

`get_db_info()`가 어떤 key 이름으로 dict를 반환하는지 먼저 출력해서 확인합니다.

일반적인 key 이름 변형(host/Host/HOST, database/db/DB, password/pwd/passwd 등)을
자동으로 pymysql 표준 형태로 변환합니다. 만약 자동 변환이 실패하면 출력된 key 목록을
보고 Cell 1-B의 `KEY_ALIASES` 매핑만 수정하면 됩니다.

In [ ]:
# ── 원본 dict 조회 ─────────────────────────────────────────
raw = config.get_db_info()
print("[get_db_info() 반환 dict]")
# password는 마스킹해서 출력
for k, v in raw.items():
    shown = "***" if any(s in k.lower() for s in ["pass", "pwd", "secret"]) else v
    print(f"  {k!r}: {shown!r}")

In [ ]:
# ── pymysql 표준 key 이름으로 정규화 ────────────────────────
# pymysql.connect()가 받는 표준 인자: host, port, user, password, database, charset

KEY_ALIASES = {
    "host":     ["host", "HOST", "Host", "hostname", "server"],
    "port":     ["port", "PORT", "Port"],
    "user":     ["user", "USER", "User", "username", "uid"],
    "password": ["password", "PASSWORD", "Password", "passwd", "pwd", "PWD"],
    "database": ["database", "DATABASE", "Database", "db", "DB", "dbname", "schema"],
    "charset":  ["charset", "CHARSET", "encoding"],
}

DB_CONFIG = {}
for std_key, aliases in KEY_ALIASES.items():
    for alias in aliases:
        if alias in raw:
            DB_CONFIG[std_key] = raw[alias]
            break

# ── 필수 키 체크 & 기본값 ──────────────────────────────────
required = ["host", "user", "password", "database"]
missing  = [k for k in required if k not in DB_CONFIG]
if missing:
    print(f"[경고] 다음 키를 자동 매핑하지 못했습니다: {missing}")
    print(f"      raw 반환 key 목록: {list(raw.keys())}")
    print(f"      KEY_ALIASES에 본인 config의 실제 key 이름을 추가해주세요.")
    raise ValueError(f"필수 DB 접속 정보 누락: {missing}")

# port는 정수여야 함 (str인 경우 변환)
if "port" in DB_CONFIG:
    DB_CONFIG["port"] = int(DB_CONFIG["port"])
else:
    DB_CONFIG["port"] = 3307  # 메모리 기본값

# charset 기본값
DB_CONFIG.setdefault("charset", "utf8mb4")

# ── 최종 확인 (password 마스킹) ──────────────────────────────
print("[pymysql 호환 DB_CONFIG]")
for k, v in DB_CONFIG.items():
    shown = "***" if k == "password" else v
    print(f"  {k}: {shown}")

In [ ]:
# ── 접속 테스트 & 라이브러리 로드 ───────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
import pymysql
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 접속 테스트
try:
    _conn = pymysql.connect(**DB_CONFIG)
    _cur = _conn.cursor()
    _cur.execute("SELECT 1")
    _cur.fetchone()
    _conn.close()
    print("[OK] DB 접속 성공")
except Exception as e:
    print(f"[FAIL] DB 접속 실패: {e}")
    raise

# ── 진단 대상 ───────────────────────────────────────────────
TICKER    = "AUPH"
BS_TABLE  = "us_balance_sheet_data"
IS_TABLE  = "us_income_statement_data"

print(f"\n진단 대상: {TICKER}")
print(f"BS 테이블: {BS_TABLE}")
print(f"IS 테이블: {IS_TABLE}")

## Cell 2 · BS / IS 원본 조회

In [ ]:
conn = pymysql.connect(**DB_CONFIG)
cur  = conn.cursor(pymysql.cursors.DictCursor)

# ── BS 테이블 컬럼 확인 ─────────────────────────────────────
cur.execute(f"SHOW COLUMNS FROM {BS_TABLE}")
bs_cols = [r["Field"] for r in cur.fetchall()]
print(f"[{BS_TABLE}] 전체 컬럼 {len(bs_cols)}개")
nwc_related = [c for c in bs_cols
               if any(k in c.lower() for k in
                      ["current", "cash", "debt", "period", "date", "ticker", "accept"])]
print(f"NWC 관련 후보: {nwc_related}")

In [ ]:
# ── BS 조회 ────────────────────────────────────────────────
select_cols = ["date", "currentAssets", "cashAndCashEquivalents",
               "totalCurrentLiabilities", "shortTermDebt"]
optional = ["period", "acceptedDate", "calendarYear", "fillingDate"]
for c in optional:
    if c in bs_cols:
        select_cols.append(c)

cols_sql = ", ".join(f"`{c}`" for c in select_cols)
cur.execute(f"""
    SELECT {cols_sql}
    FROM {BS_TABLE}
    WHERE ticker = %s
    ORDER BY date
""", (TICKER,))
bs = pd.DataFrame(cur.fetchall())

# ── IS 조회 ────────────────────────────────────────────────
cur.execute(f"SHOW COLUMNS FROM {IS_TABLE}")
is_cols = [r["Field"] for r in cur.fetchall()]

is_select = ["date", "revenue"]
for c in optional:
    if c in is_cols:
        is_select.append(c)

is_cols_sql = ", ".join(f"`{c}`" for c in is_select)
cur.execute(f"""
    SELECT {is_cols_sql}
    FROM {IS_TABLE}
    WHERE ticker = %s
    ORDER BY date
""", (TICKER,))
inc = pd.DataFrame(cur.fetchall())

conn.close()

print(f"[BS] rows = {len(bs)}, cols = {list(bs.columns)}")
print(f"[IS] rows = {len(inc)}, cols = {list(inc.columns)}")

## Cell 3 · period 분포 & date 중복 확인 (**핵심**)

In [ ]:
print("="*60)
print("[BS] period 분포")
print("="*60)
if "period" in bs.columns:
    print(bs["period"].value_counts(dropna=False))
else:
    print("period 컬럼 없음 (분기/연간 구분 불가)")

print("\n" + "="*60)
print("[IS] period 분포")
print("="*60)
if "period" in inc.columns:
    print(inc["period"].value_counts(dropna=False))
else:
    print("period 컬럼 없음")

print("\n" + "="*60)
print("[BS] date 중복")
print("="*60)
dup_bs = bs[bs.duplicated("date", keep=False)].sort_values("date")
if dup_bs.empty:
    print("중복 없음")
else:
    print(f"중복 {len(dup_bs)}행:")
    print(dup_bs.to_string())

print("\n[IS] date 중복")
dup_is = inc[inc.duplicated("date", keep=False)].sort_values("date")
if dup_is.empty:
    print("중복 없음")
else:
    print(f"중복 {len(dup_is)}행:")
    print(dup_is.to_string())

## Cell 4 · NWC 계산 & merged 데이터 전체 확인

In [ ]:
# ── NWC 계산 (DCFModel과 동일) ───────────────────────────────
for c in ["currentAssets", "cashAndCashEquivalents",
          "totalCurrentLiabilities", "shortTermDebt"]:
    bs[c] = pd.to_numeric(bs[c], errors="coerce").fillna(0)

bs["nwc"] = ((bs["currentAssets"] - bs["cashAndCashEquivalents"])
             - (bs["totalCurrentLiabilities"] - bs["shortTermDebt"]))

inc["revenue"] = pd.to_numeric(inc["revenue"], errors="coerce")

bs_cols_to_merge = ["date", "nwc"]
if "period" in bs.columns:
    bs_cols_to_merge.append("period")

inc_cols_to_merge = ["date", "revenue"]
if "period" in inc.columns:
    inc_cols_to_merge.append("period")

merged = inc[inc_cols_to_merge].merge(
    bs[bs_cols_to_merge], on="date", how="inner",
    suffixes=("_is", "_bs")
).dropna(subset=["revenue", "nwc"])

merged = merged[merged["revenue"] > 0].sort_values("date").reset_index(drop=True)
merged["ratio"] = merged["nwc"] / merged["revenue"]

print(f"merged n = {len(merged)}")
print("\n[merged 전체]")
print(merged.to_string())

In [ ]:
# ── ratio 분포 & 이상치 ─────────────────────────────────────
print("="*60)
print("NWC / Revenue ratio 분포")
print("="*60)
print(merged["ratio"].describe())

med = merged["ratio"].median()
mad = (merged["ratio"] - med).abs().median()
threshold = 3 * mad if mad > 0 else 3 * merged["ratio"].std()

merged["is_outlier"] = (merged["ratio"] - med).abs() > threshold

print(f"\nmedian = {med:.4f}, MAD = {mad:.4f}, 임계값 = {threshold:.4f}")
print(f"\n[이상치 행 ({merged['is_outlier'].sum()}개)]")
outliers = merged[merged["is_outlier"]]
if outliers.empty:
    print("없음")
else:
    print(outliers.to_string())

## Cell 5 · OLS γ vs median ratio 비교

In [ ]:
x = merged["revenue"].values
y = merged["nwc"].values

# 1) OLS (절편 포함) — 현재 DCFModel 방식
slope, intercept, r, p, se = stats.linregress(x, y)
r2 = r ** 2

# 2) 무절편 OLS
slope_no_intercept = (x * y).sum() / (x * x).sum()

# 3) median ratio
gamma_median = merged["ratio"].median()

# 4) winsorized median
ratios = merged["ratio"].copy()
lo, hi = ratios.quantile(0.05), ratios.quantile(0.95)
gamma_median_w = ratios.clip(lo, hi).median()

# 5) outlier 제거 후 OLS
clean = merged[~merged["is_outlier"]]
if len(clean) >= 10:
    slope_clean, intercept_clean, r_clean, _, _ = stats.linregress(
        clean["revenue"], clean["nwc"])
    r2_clean = r_clean ** 2
else:
    slope_clean = intercept_clean = r2_clean = np.nan

print("="*75)
print("γ 추정 방식별 비교")
print("="*75)
print(f"{'방식':<35} {'γ':>15} {'비고':<20}")
print("-"*75)
print(f"{'OLS (절편 포함, 현재 채택)':<35} {slope:>15,.4f}  R²={r2:.3f}, intercept={intercept:,.0f}")
print(f"{'OLS (무절편)':<35} {slope_no_intercept:>15,.4f}")
print(f"{'median ratio':<35} {gamma_median:>15,.4f}")
print(f"{'median ratio (5-95% winsorized)':<35} {gamma_median_w:>15,.4f}")
if not np.isnan(slope_clean):
    print(f"{'OLS (outlier 제거)':<35} {slope_clean:>15,.4f}  R²={r2_clean:.3f}")
else:
    print(f"{'OLS (outlier 제거)':<35} {'N/A':>15}")
print("="*75)

# 예측 영향 — AUPH 2026Q1 sales=60.4M 기준
sales_example = 60_424_550
print(f"\n예상 NWC @ 2026Q1 sales=${sales_example:,.0f}")
print(f"  OLS 절편포함:  ${slope * sales_example:>18,.0f}")
print(f"  OLS 무절편:    ${slope_no_intercept * sales_example:>18,.0f}")
print(f"  median:        ${gamma_median * sales_example:>18,.0f}")
print(f"  winsor median: ${gamma_median_w * sales_example:>18,.0f}")
if not np.isnan(slope_clean):
    print(f"  clean OLS:     ${slope_clean * sales_example:>18,.0f}")

## Cell 6 · 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Revenue vs NWC 산점도 + 4개 회귀선
ax = axes[0]
colors = np.where(merged["is_outlier"], "red", "steelblue")
ax.scatter(merged["revenue"], merged["nwc"], c=colors, s=60, alpha=0.7,
           edgecolors="black", linewidths=0.5)

x_range = np.linspace(0, merged["revenue"].max() * 1.1, 100)
ax.plot(x_range, slope * x_range + intercept, "r-",
        label=f"OLS w/ intercept (gamma={slope:.2f})", linewidth=2)
ax.plot(x_range, slope_no_intercept * x_range, "g--",
        label=f"OLS no-intercept (gamma={slope_no_intercept:.2f})", linewidth=2)
ax.plot(x_range, gamma_median * x_range, "b:",
        label=f"median ratio (gamma={gamma_median:.2f})", linewidth=2)
if not np.isnan(slope_clean):
    ax.plot(x_range, slope_clean * x_range + intercept_clean, "purple", linestyle="-.",
            label=f"OLS clean (gamma={slope_clean:.2f})", linewidth=2)

for _, row in merged[merged["is_outlier"]].iterrows():
    ax.annotate(str(row["date"])[:10], (row["revenue"], row["nwc"]),
                fontsize=8, color="red", xytext=(5, 5), textcoords="offset points")

ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Revenue")
ax.set_ylabel("NWC")
ax.set_title(f"{TICKER}: Revenue vs NWC (red = outliers)")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.3)

# 오른쪽: ratio 시계열
ax = axes[1]
merged_plot = merged.copy()
merged_plot["date"] = pd.to_datetime(merged_plot["date"])
ax.plot(merged_plot["date"], merged_plot["ratio"], "o-", color="steelblue")
ax.scatter(merged_plot.loc[merged_plot["is_outlier"], "date"],
           merged_plot.loc[merged_plot["is_outlier"], "ratio"],
           color="red", s=100, zorder=5, label="outlier")
ax.axhline(gamma_median, color="blue", linestyle=":",
           label=f"median = {gamma_median:.2f}")
ax.axhline(slope, color="red", linestyle="--",
           label=f"OLS gamma = {slope:.2f}")
ax.set_xlabel("Date")
ax.set_ylabel("NWC / Revenue")
ax.set_title(f"{TICKER}: NWC/Revenue ratio time series")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Cell 7 · 해석 가이드

### 결과 해석 체크리스트

**Q1. Cell 3의 period 분포에서 'FY' / 'annual' / 'TTM' 행이 BS에 섞여 있나?**
- YES → 바로 이것이 주범. `estimate_nwc_coef`에서 merge 전 분기 필터링 필요.
- NO → Q2로

**Q2. date 중복 행이 있나?**
- YES → FMP 다중 보고 이슈. `acceptedDate` 최신 기준 drop_duplicates 필요.
- NO → Q3으로

**Q3. Cell 5에서 OLS slope vs median ratio 차이가 얼마나 나나?**
- 10배 이상 → OLS가 소수 outlier에 leverage됨. median-first 전략 전환 권장.
- 2-3배 이내 → 다른 버그 의심.

**Q4. Cell 6 왼쪽 그래프에서 빨간 점(outlier)이 회귀선을 크게 끌고 있나?**
- YES → leverage point 확정. clean OLS 또는 robust regression 필요.